# Phase 7 — Persistence Baseline

**What:** use current-hour PM2.5 as the prediction of next-hour PM2.5.

**Why:** a machine-learning model should be compared with a simple forecasting reference.

**Decision:** evaluate the baseline on the same walk-forward validation folds used for Phase 8. The final 2017 holdout is not inspected here for model-selection decisions.

## 1. Load walk-forward validation files

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src" / "project_config.py").is_file():
        PROJECT_ROOT = candidate
        break
sys.path.insert(0, str(PROJECT_ROOT))
from src.project_config import *

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

manifest_path = SPLIT_DIR / "walk_forward_manifest.csv"
if not manifest_path.exists():
    raise FileNotFoundError("Run 06_data_split.ipynb first.")

manifest = pd.read_csv(manifest_path)


## 2. Calculate baseline metrics for each validation fold

In [2]:
rows = []

for _, item in manifest.iterrows():
    validation = pd.read_csv(SPLIT_DIR / item["validation_file"])
    y_true = validation[TARGET].to_numpy()
    y_pred = validation["PM2.5"].to_numpy()

    rows.append([
        item["fold"],
        mean_absolute_error(y_true, y_pred),
        np.sqrt(mean_squared_error(y_true, y_pred)),
        r2_score(y_true, y_pred)
    ])

results = pd.DataFrame(rows, columns=["fold", "MAE", "RMSE", "R2"])
results.loc["mean"] = ["Mean", results["MAE"].mean(), results["RMSE"].mean(), results["R2"].mean()]
display(results.round(4))

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
results.to_csv(AUDIT_DIR / "phase7_walk_forward_persistence_baseline.csv", index=False)
print("Created:", AUDIT_DIR / "phase7_walk_forward_persistence_baseline.csv")

,fold,MAE,RMSE,R2
0,fold_1,10.4255,18.9954,0.9419
1,fold_2,10.3956,19.5011,0.9465
2,fold_3,9.5789,18.4193,0.9411
mean,Mean,10.1333,18.9720,0.9432


Created: /Users/ruchi/Downloads/phase12_fixed/data/audit/phase7_walk_forward_persistence_baseline.csv


## Baseline decision

Use the mean walk-forward baseline metrics as the minimum development reference. An ML model should be judged on the same validation folds and should provide evidence of improvement that is reasonably consistent across periods, not only in one fold.

The final 2017 holdout remains protected.